In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder\
        .appName("e-commerce_analysis")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "1g")\
        .getOrCreate()

In [ ]:
# data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-subset-10p-enriched.csv"
data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-enriched.csv"
output_dir = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/parquet/"

In [ ]:
schema = StructType([
    StructField("timestamp", TimestampType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("category_id", LongType()),
    StructField("category_code", StringType()),
    StructField("brand", StringType()),
    StructField("price", FloatType()),
    StructField("user_id", LongType()),
    StructField("user_session", StringType()),
    StructField("name", StringType()),
    StructField("username", StringType()),
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("sex", StringType()),
    StructField("product_name", StringType()),
    StructField("product_description", StringType())
])

In [ ]:
df = spark.read.format("csv")\
    .schema(schema)\
    .option("header", "true")\
    .load(data_path)\
    .select("event_time", "event_type", "product_id", "price", "user_id", "user_session")
print("Number of partitions:", df.rdd.getNumPartitions())
df.cache()

In [ ]:
df.orderBy(["user_session", "product_id"]).take(30)

In [ ]:
df.createOrReplaceTempView("event_log")

In [ ]:
# TODO Create one table containing all view records
view_df = spark.sql("SELECT DISTINCT * FROM event_log WHERE event_type = 'view'")

In [ ]:
# TODO Create one table containing all cart & purchase records
engagement_df = spark.sql("SELECT DISTINCT * FROM event_log WHERE event_type != 'view'")

In [ ]:
# TODO Join the two tables on user_session and product_id
view_df.join(engagement_df, on=["user_session", "product_id"], how="inner").orderBy("user_session").show(20)

In [ ]:
user_sessions = df.orderBy(["user_id", "user_session", "event_time"])
user_sessions.show()

In [ ]:
# Coalesce into fewer partitions
df.coalesce(5).write.partitionBy("timestamp").parquet(path=output_dir, mode="overwrite")

In [ ]:
spark.stop()